# LangGraph — ترکیب با LLM

## Outline
* Chatbot ساده با StateGraph + LLM
* افزودن `add_messages` reducer
* Agent با Tool-Calling از صفر (بدون `create_agent`)
* مقایسه StateGraph خام vs `create_agent`



In [2]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

## ۱. Chatbot ساده با StateGraph

ابتدا یادآوری: در نوت‌بوک قبل State فقط `int` و `str` داشت.  
حالا State شامل **لیست پیام‌ها** می‌شود.

### مشکل: Reducer برای messages

```
بدون reducer:
  Node A → {"messages": [msg1]}      # جایگزین می‌شود
  Node B → {"messages": [msg2]}      # msg1 از بین می‌رود!

با add_messages reducer:
  Node A → {"messages": [msg1]}      # اضافه می‌شود
  Node B → {"messages": [msg2]}      # msg1 + msg2 = هر دو نگه داشته می‌شوند ✓
```

In [5]:
from typing import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

# ── ۱. State با add_messages reducer ────────────────────
class ChatState(TypedDict):
    # add_messages: پیام‌های جدید اضافه می‌شوند (نه جایگزین)
    messages: Annotated[list[BaseMessage], add_messages]

# ── ۲. LLM ──────────────────────────────────────────────
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

# ── ۳. Node: فقط یک تابع که LLM صدا می‌زند ─────────────
def call_llm(state: ChatState) -> dict:
    """Node: پیام‌های State را به LLM می‌دهد و پاسخ را اضافه می‌کند"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # add_messages آن را اضافه می‌کند

# ── ۴. ساخت گراف ────────────────────────────────────────
builder = StateGraph(ChatState)
builder.add_node("llm", call_llm)
builder.add_edge(START, "llm")
builder.add_edge("llm", END)

memory = InMemorySaver()
chatbot = builder.compile(checkpointer=memory)

# ── ۵. تست با حافظه ─────────────────────────────────────
config = {"configurable": {"thread_id": "test_1"}}

# پیام اول
r1 = chatbot.invoke(
    {"messages": [HumanMessage(content="سلام! اسم من علی است.")]},
    config
)
print("پیام ۱:", r1["messages"][-1].content)

# پیام دوم — chatbot باید اسم را یادش باشد
r2 = chatbot.invoke(
    {"messages": [HumanMessage(content="اسم من چیست؟")]},
    config
)
print("\nپیام ۲:", r2["messages"][-1].content)

پیام ۱: سلام علی! چطور می‌توانم به شما کمک کنم؟

پیام ۲: اسم شما علی است. چطور می‌توانم به شما کمک کنم؟


## ۲. Agent با Tool-Calling از صفر

حالا همان Agent pattern را که `create_agent` می‌سازد، خودمان با StateGraph می‌سازیم.

```
ReAct Loop در LangGraph:

  START
    │
    ▼
  [agent]  ← LLM تصمیم می‌گیرد
    │
    ├─ tool_calls وجود دارد → [tools]  ← tools اجرا می‌شوند
    │                              │
    │                              └──────────────────────┐
    │                                                     │
    │◄────────────────────────────────────────────────────┘
    │
    └─ tool_calls ندارد → END
```

In [ ]:
from typing import TypedDict, Annotated, Literal
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# ── ۱. Toolها ────────────────────────────────────────────
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers"""
    return a + b

tools = [multiply, add]

# ── ۲. LLM با tools bind شده ────────────────────────────
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# ── ۳. State ─────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# ── ۴. Nodeها ────────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    """LLM تصمیم می‌گیرد: tool بزند یا جواب بدهد"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# ToolNode آماده از LangGraph — toolها را خودکار اجرا می‌کند
tool_node = ToolNode(tools)

# ── ۵. Router ────────────────────────────────────────────
def should_use_tool(state: AgentState) -> Literal["tools", "__end__"]:
    """اگر LLM tool_call داشت → tools، وگرنه تمام"""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "__end__"

# ── ۶. ساخت گراف ────────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_use_tool,
    {"tools": "tools", "__end__": END}
)

builder.add_edge("tools", "agent")  # ← بعد از tool → دوباره به agent

react_agent = builder.compile()

# ── ۷. تست ──────────────────────────────────────────────
print("=== Agent Test ===")
result = react_agent.invoke({
    "messages": [HumanMessage(content="حاصل (۳ × ۴) + ۵ چند می‌شود؟")]
})
print(result["messages"][-1].content)

### مشاهده مراحل اجرا (Streaming)

In [ ]:
print("=== مراحل اجرا ===")
for step in react_agent.stream(
    {"messages": [HumanMessage(content="حاصل (۳ × ۴) + ۵ چند می‌شود؟")]},
    stream_mode="updates"  # هر update را جداگانه نشان می‌دهد
):
    node_name = list(step.keys())[0]
    msgs = step[node_name]["messages"]
    last = msgs[-1]
    
    if node_name == "agent":
        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                print(f"  [agent → tool_call]  {tc['name']}({tc['args']})")
        else:
            print(f"  [agent → پاسخ]  {last.content}")
    elif node_name == "tools":
        print(f"  [tools → نتیجه]  {last.content}")

## ۳. مقایسه: StateGraph خام vs `create_agent`

```
create_agent (نوت‌بوک‌های قبلی):
──────────────────────────────
  agent = create_agent(model=llm, tools=[...], system_prompt="...")
  # ← همین agent بالا را می‌سازد، اما شما کنترلی روی گراف ندارید

StateGraph (این نوت‌بوک):
──────────────────────────────
  builder = StateGraph(AgentState)
  builder.add_node("agent", agent_node)
  builder.add_node("tools", tool_node)
  builder.add_conditional_edges(...)
  # ← شما هر قدم را کنترل می‌کنید
```

| | `create_agent` | StateGraph خام |
|---|---|---|
| **سرعت** | سریع | کندتر |
| **کنترل** | محدود | کامل |
| **State سفارشی** | با AgentState | هر TypedDict |
| **مسیریابی پیچیده** | محدود | آزاد |
| **کاربرد** | agent معمولی | workflow پیچیده |

> **قانون کلی**: با `create_agent` شروع کن. وقتی به کنترل بیشتر نیاز داشتی، به StateGraph برو.

## ۴. مثال کاربردی: گراف بررسی متن

یک pipeline که متن را بررسی می‌کند:  
اگر فارسی بود ترجمه می‌کند، سپس خلاصه می‌نویسد.

In [ ]:
from typing import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

class TextState(TypedDict):
    text: str
    language: str       # "fa" یا "en"
    english_text: str   # ترجمه (اگر لازم باشد)
    summary: str

# ── Node: تشخیص زبان ─────────────────────────────────────
def detect_language(state: TextState) -> dict:
    prompt = f"فقط 'fa' یا 'en' جواب بده. زبان این متن چیست:\n{state['text']}"
    lang = llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
    lang = "fa" if "fa" in lang else "en"
    print(f"  [detect_language]  زبان: {lang}")
    return {"language": lang}

# ── Node: ترجمه ──────────────────────────────────────────
def translate_to_english(state: TextState) -> dict:
    prompt = f"این متن را به انگلیسی ترجمه کن:\n{state['text']}"
    translated = llm.invoke([HumanMessage(content=prompt)]).content
    print(f"  [translate]  ترجمه شد")
    return {"english_text": translated}

# ── Node: خلاصه ──────────────────────────────────────────
def summarize(state: TextState) -> dict:
    source = state["english_text"] if state["english_text"] else state["text"]
    prompt = f"این متن را در یک جمله خلاصه کن:\n{source}"
    summary = llm.invoke([HumanMessage(content=prompt)]).content
    return {"summary": summary}

# ── Router ───────────────────────────────────────────────
def route_language(state: TextState) -> Literal["translate", "summarize"]:
    return "translate" if state["language"] == "fa" else "summarize"

# ── ساخت گراف ───────────────────────────────────────────
builder = StateGraph(TextState)
builder.add_node("detect", detect_language)
builder.add_node("translate", translate_to_english)
builder.add_node("summarize", summarize)

builder.add_edge(START, "detect")
builder.add_conditional_edges(
    "detect",
    route_language,
    {"translate": "translate", "summarize": "summarize"}
)
builder.add_edge("translate", "summarize")
builder.add_edge("summarize", END)

pipeline = builder.compile()

# ── تست با متن فارسی ─────────────────────────────────────
print("=== متن فارسی ===")
res = pipeline.invoke({
    "text": "هوش مصنوعی در حال تغییر دنیا است و نقش مهمی در آینده انسان‌ها خواهد داشت.",
    "language": "", "english_text": "", "summary": ""
})
print(f"خلاصه: {res['summary']}")

print("\n=== متن انگلیسی ===")
res = pipeline.invoke({
    "text": "Artificial intelligence is transforming industries and creating new opportunities.",
    "language": "", "english_text": "", "summary": ""
})
print(f"خلاصه: {res['summary']}")

```
ساختار گراف:

  START → [detect]
               │
               ├─ فارسی → [translate] → [summarize] → END
               └─ انگلیسی ───────────→ [summarize] → END
```

## جمع‌بندی کل دوره LangGraph

```
نوت‌بوک ۱ (08_09a) — مفاهیم پایه:
  ├── State (TypedDict)
  ├── Node (تابع Python)
  ├── Edge (مستقیم)
  ├── Conditional Edge (شرطی)
  ├── Reducer (Annotated + operator.add)
  ├── Loop (حلقه)
  └── Checkpointing

نوت‌بوک ۲ (08_09b) — ترکیب با LLM:
  ├── add_messages reducer
  ├── Chatbot با حافظه
  ├── ReAct Agent از صفر (ToolNode)
  ├── Streaming مراحل
  └── Pipeline چند مرحله‌ای

نوت‌بوک‌های بعدی (08_09، 08_10، 08_11):
  ├── State سفارشی در create_agent
  ├── Multi-Agent
  └── Human-in-the-Loop
```